# GLIME 05: STL-10 Classification + LIME Explanations

This notebook upgrades the original CIFAR-10 notebook from folder `04` into a stronger STL-10 workflow.

What this notebook does:

- loads the labeled STL-10 training and test splits
- loads the unlabeled STL-10 split so the dataset is represented completely
- trains a stronger CNN baseline for STL-10
- evaluates the model on validation and test data
- explains one prediction with standard LIME
- explains the same prediction with a custom Gaussian-noise version of LIME

Why this notebook is different from the original:

- STL-10 images are larger than CIFAR-10 images, so the model and training setup need to be adjusted
- STL-10 also includes 100,000 unlabeled images, which are important context for the dataset even when the baseline below remains supervised
- the explanation sections are commented more clearly so each step is easier to follow

In [ ]:
# =========================
# 0. Imports
# =========================

import copy
import random

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms

from lime import lime_image
from skimage.segmentation import slic, mark_boundaries
from sklearn.linear_model import Ridge

In [ ]:
# =========================
# 1. Setup
# =========================

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

class_names = [
    "airplane", "bird", "car", "cat", "deer",
    "dog", "horse", "monkey", "ship", "truck"
]

num_classes = len(class_names)
data_root = "./data"
batch_size_train = 64
batch_size_eval = 128
num_workers = 2

## STL-10 Split Overview

STL-10 contains three useful splits:

- `train`: labeled images for supervised learning
- `test`: labeled images for final evaluation
- `unlabeled`: a much larger pool of images without class labels

The model below is still a supervised baseline, so it trains only on labeled data.  
We still load the unlabeled split because it is part of what makes STL-10 different from CIFAR-10, and it is useful for future self-supervised or semi-supervised extensions.

In [ ]:
# =========================
# 2. Transforms and dataset loading
# =========================

# Training data gets light augmentation to help the model generalize
# from STL-10's relatively small labeled training split.
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(96, padding=8, padding_mode="reflect"),
    transforms.ToTensor(),
])

# Evaluation and explanation images stay unaugmented.
eval_transform = transforms.Compose([
    transforms.ToTensor(),
])

full_train_dataset = datasets.STL10(
    root=data_root,
    split="train",
    download=True,
    transform=train_transform
)

eval_train_dataset = datasets.STL10(
    root=data_root,
    split="train",
    download=True,
    transform=eval_transform
)

test_dataset = datasets.STL10(
    root=data_root,
    split="test",
    download=True,
    transform=eval_transform
)

unlabeled_dataset = datasets.STL10(
    root=data_root,
    split="unlabeled",
    download=True,
    transform=eval_transform
)

print("Labeled train examples:", len(full_train_dataset))
print("Test examples:", len(test_dataset))
print("Unlabeled examples:", len(unlabeled_dataset))

In [ ]:
# =========================
# 3. Build train/validation splits
# =========================

# STL-10 has a small labeled training split, so holding out a validation
# set helps us track generalization and keep the test set untouched until
# the final evaluation step.
train_size = int(0.9 * len(full_train_dataset))
val_size = len(full_train_dataset) - train_size

split_generator = torch.Generator().manual_seed(42)

train_dataset, val_dataset = random_split(
    full_train_dataset,
    [train_size, val_size],
    generator=split_generator
)

# For choosing explanation images later, it is convenient to have the
# matching raw, unaugmented validation subset as well.
_, raw_val_dataset = random_split(
    eval_train_dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size_train,
    shuffle=True,
    num_workers=num_workers
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size_eval,
    shuffle=False,
    num_workers=num_workers
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size_eval,
    shuffle=False,
    num_workers=num_workers
)

print("Train split size:", len(train_dataset))
print("Validation split size:", len(val_dataset))

In [ ]:
# =========================
# 4. Visualize a few labeled and unlabeled samples
# =========================

def show_examples(dataset, title, rows=2, cols=5):
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 2.2, rows * 2.2))
    fig.suptitle(title, fontsize=14)

    for idx, ax in enumerate(axes.flat):
        image, label = dataset[idx]
        image_np = image.permute(1, 2, 0).numpy()

        ax.imshow(image_np)
        if isinstance(label, (int, np.integer)) and label >= 0:
            ax.set_title(class_names[label], fontsize=9)
        else:
            ax.set_title("unlabeled", fontsize=9)
        ax.axis("off")

    plt.tight_layout()
    plt.show()


show_examples(eval_train_dataset, "Sample labeled STL-10 training images")
show_examples(unlabeled_dataset, "Sample unlabeled STL-10 images")

In [ ]:
# =========================
# 5. Improved CNN for STL-10
# =========================

class ImprovedSTL10CNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((3, 3)),
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 3 * 3, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.4),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


model = ImprovedSTL10CNN(num_classes=num_classes).to(device)
print(model)

In [ ]:
# =========================
# 6. Training and evaluation helpers
# =========================

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)


def run_epoch(model, loader, optimizer=None):
    is_training = optimizer is not None
    model.train(is_training)

    total_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        if is_training:
            optimizer.zero_grad()

        with torch.set_grad_enabled(is_training):
            logits = model(images)
            loss = criterion(logits, labels)

            if is_training:
                loss.backward()
                optimizer.step()

        total_loss += loss.item() * images.size(0)
        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    average_loss = total_loss / total
    accuracy = correct / total
    return average_loss, accuracy


@torch.no_grad()
def collect_predictions(model, loader):
    model.eval()

    all_preds = []
    all_labels = []

    for images, labels in loader:
        images = images.to(device)
        logits = model(images)
        preds = logits.argmax(dim=1).cpu().numpy()

        all_preds.extend(preds.tolist())
        all_labels.extend(labels.numpy().tolist())

    return np.array(all_preds), np.array(all_labels)

In [ ]:
# =========================
# 7. Train the model
# =========================

num_epochs = 12
history = {
    "train_loss": [],
    "train_acc": [],
    "val_loss": [],
    "val_acc": [],
}

best_val_acc = 0.0
best_state = copy.deepcopy(model.state_dict())

for epoch in range(num_epochs):
    train_loss, train_acc = run_epoch(model, train_loader, optimizer=optimizer)
    val_loss, val_acc = run_epoch(model, val_loader, optimizer=None)
    scheduler.step()

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_state = copy.deepcopy(model.state_dict())

    current_lr = optimizer.param_groups[0]["lr"]
    print(
        f"Epoch {epoch + 1:02d}/{num_epochs} | "
        f"train loss: {train_loss:.4f} | train acc: {train_acc:.4f} | "
        f"val loss: {val_loss:.4f} | val acc: {val_acc:.4f} | "
        f"lr: {current_lr:.6f}"
    )

model.load_state_dict(best_state)
print(f"Best validation accuracy: {best_val_acc:.4f}")

In [ ]:
# =========================
# 8. Plot training curves
# =========================

epochs = np.arange(1, num_epochs + 1)

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(epochs, history["train_loss"], marker="o", label="Train loss")
plt.plot(epochs, history["val_loss"], marker="o", label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training and validation loss")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(epochs, history["train_acc"], marker="o", label="Train accuracy")
plt.plot(epochs, history["val_acc"], marker="o", label="Validation accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training and validation accuracy")
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# =========================
# 9. Final test evaluation
# =========================

test_loss, test_acc = run_epoch(model, test_loader, optimizer=None)
print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_acc:.4f}")

test_preds, test_labels = collect_predictions(model, test_loader)

per_class_accuracy = {}
for class_index, class_name in enumerate(class_names):
    class_mask = test_labels == class_index
    class_accuracy = (test_preds[class_mask] == test_labels[class_mask]).mean()
    per_class_accuracy[class_name] = class_accuracy

for class_name, class_accuracy in per_class_accuracy.items():
    print(f"{class_name:>8}: {class_accuracy:.4f}")

## LIME Setup

LIME explains one prediction by perturbing parts of an image and measuring how the model's output changes.

For image explanations, there are three important pieces:

- a prediction function that accepts batches of NumPy images
- a segmentation function that divides the image into superpixels
- an example image we want to explain

In [ ]:
# =========================
# 10. Prediction wrapper for LIME
# =========================

def predict_fn(images):
    """
    LIME passes a batch of images as a NumPy array with shape:
    (batch, height, width, channels)

    This wrapper converts that batch into the PyTorch format expected
    by the CNN and returns class probabilities.
    """

    model.eval()

    images = np.array(images).astype(np.float32)

    # LIME often uses pixel values in the 0-255 range, so we rescale
    # back to 0-1 before feeding the images to the model.
    if images.max() > 1.0:
        images = images / 255.0

    images = torch.tensor(images).permute(0, 3, 1, 2).float().to(device)

    with torch.no_grad():
        logits = model(images)
        probs = F.softmax(logits, dim=1)

    return probs.cpu().numpy()

In [ ]:
# =========================
# 11. Choose an explanation image
# =========================

def find_correct_example(dataset):
    """
    Pick the first validation example that the model classifies correctly.
    Using a correct example makes the explanation easier to interpret.
    """

    model.eval()

    for idx in range(len(dataset)):
        image_tensor, label = dataset[idx]
        probs = predict_fn(np.array([(image_tensor.permute(1, 2, 0).numpy() * 255).astype(np.uint8)]))[0]
        pred = int(np.argmax(probs))

        if pred == label:
            return idx, image_tensor, label, probs

    # Fallback: if no correct example is found, just use the first item.
    image_tensor, label = dataset[0]
    probs = predict_fn(np.array([(image_tensor.permute(1, 2, 0).numpy() * 255).astype(np.uint8)]))[0]
    return 0, image_tensor, label, probs


example_idx, image_tensor, true_label, probs = find_correct_example(raw_val_dataset)

image = image_tensor.permute(1, 2, 0).numpy()
image_uint8 = (image * 255).astype(np.uint8)
pred_label = int(np.argmax(probs))

plt.figure(figsize=(4, 4))
plt.imshow(image_uint8)
plt.title(
    f"Index: {example_idx} | True: {class_names[true_label]} | "
    f"Pred: {class_names[pred_label]}"
)
plt.axis("off")
plt.show()

top3_indices = np.argsort(probs)[-3:][::-1]
for rank, class_index in enumerate(top3_indices, start=1):
    print(f"Top {rank}: {class_names[class_index]} -> {probs[class_index]:.4f}")

In [ ]:
# =========================
# 12. Shared segmentation function
# =========================

def segmentation_fn(image):
    """
    SLIC groups nearby pixels into superpixels. LIME explains the model
    at the superpixel level instead of at the raw-pixel level because
    that is usually easier for humans to interpret.
    """

    return slic(
        image,
        n_segments=70,
        compactness=12,
        sigma=1,
        start_label=0
    )


segments = segmentation_fn(image_uint8)

plt.figure(figsize=(5, 5))
plt.imshow(
    mark_boundaries(
        image_uint8 / 255.0,
        segments,
        color=(1, 0, 0),
        mode="thick"
    )
)
plt.title("Superpixel segmentation before explanation")
plt.axis("off")
plt.show()

print("Number of segments:", len(np.unique(segments)))

In [ ]:
# =========================
# 13. Standard LIME explanation
# =========================

explainer = lime_image.LimeImageExplainer(random_state=42)

# Standard LIME creates perturbed versions of the image by turning
# superpixels on and off, then fits a simple local surrogate model
# around the selected example.
standard_explanation = explainer.explain_instance(
    image_uint8,
    predict_fn,
    top_labels=1,
    hide_color=0,
    num_samples=1000,
    segmentation_fn=segmentation_fn
)

standard_label = standard_explanation.top_labels[0]

temp, mask = standard_explanation.get_image_and_mask(
    standard_label,
    positive_only=False,
    num_features=12,
    hide_rest=False
)

plt.figure(figsize=(5, 5))
plt.imshow(
    mark_boundaries(
        temp / 255.0,
        mask,
        color=(1, 0, 0),
        mode="thick"
    )
)
plt.title(f"Standard LIME for: {class_names[standard_label]}")
plt.axis("off")
plt.show()

In [ ]:
# =========================
# 14. Standard LIME positive-only view
# =========================

# positive_only=True keeps only the superpixels that support the chosen
# class, which usually makes the explanation easier to read.
temp_pos, mask_pos = standard_explanation.get_image_and_mask(
    standard_label,
    positive_only=True,
    num_features=12,
    hide_rest=False
)

plt.figure(figsize=(5, 5))
plt.imshow(
    mark_boundaries(
        temp_pos / 255.0,
        mask_pos,
        color=(0, 1, 0),
        mode="thick"
    )
)
plt.title("Standard LIME: positive supporting superpixels")
plt.axis("off")
plt.show()

## Custom Gaussian-Noise LIME

The custom version below keeps the same local-surrogate idea as LIME, but changes how perturbations are created.

Instead of hiding a superpixel completely, it adds Gaussian noise to the superpixels that are switched off.  
That lets us ask a slightly different question:

> Which regions matter most when we locally corrupt the image rather than blanking regions out?

In [ ]:
# =========================
# 15. Gaussian-noise perturbation function
# =========================

def perturb_with_noise(image, segments, mask, sigma_noise=30):
    """
    mask[i] = 1 means keep that segment unchanged
    mask[i] = 0 means corrupt that segment with Gaussian noise

    This produces a softer perturbation than replacing a region with a
    solid color, which can sometimes preserve more natural-looking images.
    """

    perturbed = image.astype(np.float32).copy()
    unique_segments = np.unique(segments)

    noise = np.random.normal(
        loc=0,
        scale=sigma_noise,
        size=image.shape
    )

    for i, seg_val in enumerate(unique_segments):
        if mask[i] == 0:
            region = segments == seg_val
            perturbed[region] += noise[region]

    return np.clip(perturbed, 0, 255).astype(np.uint8)

In [ ]:
# =========================
# 16. Show one Gaussian-noise perturbation
# =========================

num_segments = len(np.unique(segments))
example_mask = np.random.randint(0, 2, num_segments)

example_perturbed = perturb_with_noise(
    image_uint8,
    segments,
    example_mask,
    sigma_noise=35
)

plt.figure(figsize=(10, 5))

plt.subplot(1, 2, 1)
plt.imshow(mark_boundaries(image_uint8 / 255.0, segments, color=(1, 0, 0), mode="thick"))
plt.title("Original image with superpixels")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(mark_boundaries(example_perturbed / 255.0, segments, color=(1, 0, 0), mode="thick"))
plt.title("One Gaussian-noise perturbation")
plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# =========================
# 17. Gaussian-noise LIME sampling
# =========================

def gaussian_noise_lime(
    image,
    num_samples=1000,
    sigma_noise=35,
    sigma_kernel=10
):
    """
    Build a local explanation dataset.

    For each sampled binary mask:
    - keep some superpixels unchanged
    - corrupt the others with Gaussian noise
    - run the classifier on the perturbed image
    - assign a locality weight so perturbations closer to the original
      image matter more to the surrogate model
    """

    segments = segmentation_fn(image)
    unique_segments = np.unique(segments)
    num_segments = len(unique_segments)

    data = []
    preds = []
    sample_weights = []

    for _ in range(num_samples):
        mask = np.random.randint(0, 2, num_segments)

        perturbed = perturb_with_noise(
            image,
            segments,
            mask,
            sigma_noise=sigma_noise
        )

        pred = predict_fn(np.array([perturbed]))[0]

        # Distance here is just the number of superpixels we changed.
        distance = np.sum(mask == 0)

        # Samples closer to the original image get larger weights.
        weight = np.exp(-(distance ** 2) / (sigma_kernel ** 2))

        data.append(mask)
        preds.append(pred)
        sample_weights.append(weight)

    return (
        segments,
        np.array(data),
        np.array(preds),
        np.array(sample_weights)
    )

In [ ]:
# =========================
# 18. Fit the Gaussian-noise surrogate model
# =========================

gn_segments, gn_data, gn_preds, gn_sample_weights = gaussian_noise_lime(
    image_uint8,
    num_samples=1000,
    sigma_noise=35,
    sigma_kernel=10
)

target_class = int(np.argmax(predict_fn(np.array([image_uint8]))[0]))
y = gn_preds[:, target_class]

# A simple linear surrogate gives one importance weight per superpixel.
surrogate = Ridge(alpha=1.0)
surrogate.fit(
    gn_data,
    y,
    sample_weight=gn_sample_weights
)

gn_weights = surrogate.coef_
print("Target class for Gaussian-noise LIME:", class_names[target_class])

In [ ]:
# =========================
# 19. Visualize Gaussian-noise LIME as a heatmap
# =========================

importance = np.zeros(gn_segments.shape)

for i, seg_val in enumerate(np.unique(gn_segments)):
    importance[gn_segments == seg_val] = gn_weights[i]

importance_norm = (
    importance - importance.min()
) / (
    importance.max() - importance.min() + 1e-8
)

plt.figure(figsize=(6, 5))
plt.imshow(image_uint8)
plt.imshow(importance_norm, cmap="jet", alpha=0.55)
plt.imshow(
    mark_boundaries(
        np.zeros_like(image_uint8) / 255.0,
        gn_segments,
        color=(1, 1, 1),
        mode="thick"
    ),
    alpha=0.35
)
plt.colorbar()
plt.title(f"Gaussian-noise LIME heatmap: {class_names[target_class]}")
plt.axis("off")
plt.show()

In [ ]:
# =========================
# 20. Side-by-side comparison
# =========================

plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.imshow(image_uint8)
plt.title("Original image")
plt.axis("off")

plt.subplot(1, 3, 2)
plt.imshow(mark_boundaries(temp / 255.0, mask, color=(1, 0, 0), mode="thick"))
plt.title("Standard LIME")
plt.axis("off")

plt.subplot(1, 3, 3)
plt.imshow(image_uint8)
plt.imshow(importance_norm, cmap="jet", alpha=0.55)
plt.imshow(
    mark_boundaries(
        np.zeros_like(image_uint8) / 255.0,
        gn_segments,
        color=(1, 1, 1),
        mode="thick"
    ),
    alpha=0.35
)
plt.title("Gaussian-noise LIME")
plt.axis("off")

plt.tight_layout()
plt.show()

## Next Extension Ideas

If you want to push this notebook further later, the most STL-10-specific improvement would be to use the `unlabeled` split for self-supervised or semi-supervised pretraining before the supervised training stage.